## Correctness check for segsum

In [1]:
import json
from dataclasses import dataclass
from typing import Iterable, NamedTuple, TypeAlias, cast

import torch
import torch.nn.functional as F
from einops import rearrange, repeat
from torch import LongTensor, Tensor, nn

Device: TypeAlias = str | torch.device | None


def segsum(x: Tensor, device: Device = None) -> Tensor:
    """Stable segment sum calculation.

    `exp(segsum(A))` produces a 1-semiseparable matrix, which is equivalent to a scalar SSM.

    Source: https://github.com/state-spaces/mamba/blob/219f03c840d5a44e7d42e4e728134834fddccf45/mamba_ssm/modules/ssd_minimal.py#L23-L32
    """
    T = x.size(-1)
    x = repeat(x, "... d -> ... d e", e=T)
    mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device=device), diagonal=-1)
    x = x.masked_fill(~mask, 0)
    x_segsum = torch.cumsum(x, dim=-2)
    mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device=device), diagonal=0)
    x_segsum = x_segsum.masked_fill(~mask, -torch.inf)
    return x_segsum

In [2]:
x = torch.Tensor([1,2,3,4])
segsum(x)

tensor([[0., -inf, -inf, -inf],
        [2., 0., -inf, -inf],
        [5., 3., 0., -inf],
        [9., 7., 4., 0.]])

In [3]:
x = torch.Tensor([5,4,3,2,1])
segsum(x)

tensor([[ 0., -inf, -inf, -inf, -inf],
        [ 4.,  0., -inf, -inf, -inf],
        [ 7.,  3.,  0., -inf, -inf],
        [ 9.,  5.,  2.,  0., -inf],
        [10.,  6.,  3.,  1.,  0.]])

In [ ]:
import jax

# More stable segment sum calculation.
from mamba2 import segsum as segsum_jax


In [5]:
x = jax.numpy.array([1,2,3,4])

segsum_jax(x)

Array([[  0., -inf, -inf, -inf],
       [  2.,   0., -inf, -inf],
       [  5.,   3.,   0., -inf],
       [  9.,   7.,   4.,   0.]], dtype=float32, weak_type=True)

In [6]:
x = jax.numpy.array([5,4,3,2,1])

segsum_jax(x)

Array([[  0., -inf, -inf, -inf, -inf],
       [  4.,   0., -inf, -inf, -inf],
       [  7.,   3.,   0., -inf, -inf],
       [  9.,   5.,   2.,   0., -inf],
       [ 10.,   6.,   3.,   1.,   0.]], dtype=float32, weak_type=True)